# CSC 4110: MACHINE LEARNING

**Group:** G9 – Staff Attrition Prediction

## Module 10: Deployment and Responsible AI

### Task 9.1: Model Packaging and Web App

**Task Owner:** Jack Otuko (Member 4)

**Dataset:** IBM HR Analytics Employee Attrition Dataset

**Classification Target:** Attrition

**Objective:**

To package the selected staff attrition classification model into a complete machine learning pipeline, save and reload the model, and develop an interactive Streamlit web application that estimates employee attrition risk and provides suggested supportive HR conversations.

**Main Deliverables:**
1. Complete trained pipeline (`final_pipe`).
2. Saved model (`models/G09_model.joblib`).
3. Streamlit application (`app/app.py`).
4. Application dependencies (`app/requirements.txt`).
5. Screenshot of the running application (`outputs/app_screenshot.png`).

## 1. Import Libraries and Load Dataset

In this section, we import the required Python libraries, connect Google Drive, and load the IBM HR Analytics Employee Attrition dataset.

We will prepare the dataset by removing irrelevant columns and separating the input features from the target variable (`Attrition`).

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import os
import json
import joblib

from google.colab import drive
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier

# Connect Google Drive
drive.mount('/content/drive')

# Load the dataset
file_path = "/content/drive/MyDrive/CSC4110_G09/data/WA_Fn-UseC_-HR-Employee-Attrition.csv"
df = pd.read_csv(file_path)

# Display the first five rows
df.head()

Mounted at /content/drive


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


## 2. Data Preparation

We prepare the dataset for model training by removing irrelevant columns, separating the independent variables (features) from the dependent variable (target), and identifying numerical and categorical features.

The target variable is `Attrition`, where `No` is encoded as 0 and `Yes` is encoded as 1.

In [ ]:
# Remove irrelevant columns
df = df.drop(columns=[
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
])

# Separate features and target
X = df.drop(columns=["Attrition"])
y = df["Attrition"].map({"No": 0, "Yes": 1})

# Identify numerical and categorical features
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

# Display dataset information
print("Dataset shape:", X.shape)
print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("\nTarget distribution:")
print(y.value_counts())

Dataset shape: (1470, 30)
Numerical features: 23
Categorical features: 7

Target distribution:
Attrition
0    1233
1     237
Name: count, dtype: int64


## 3. Build the Preprocessing Pipeline

We use a `ColumnTransformer` to handle numerical and categorical features separately.

- Numerical features are passed through unchanged.
- Categorical features are converted into numerical form using `OneHotEncoder`.
- `handle_unknown="ignore"` allows the pipeline to handle categories that were not encountered during training.

This ensures that preprocessing and model training can be saved together as one complete pipeline.

In [ ]:
# Create the preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


## 4. Build and Train the Final Model

We use `GradientBoostingClassifier` to build the final attrition classification model.

The preprocessing stage and classifier are combined into one Scikit-learn `Pipeline` called `final_pipe`.

The pipeline is then trained using the prepared dataset.

In [ ]:
# Build the complete machine learning pipeline
final_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])

# Train the final pipeline
final_pipe.fit(X, y)

print("Final pipeline trained successfully.")
print("Training records:", X.shape[0])
print("Number of input features:", X.shape[1])

Final pipeline trained successfully.
Training records: 1470
Number of input features: 30


## 5. Save the Trained Model

We save the complete trained pipeline using `joblib`.

The saved pipeline contains both the preprocessing steps and the Gradient Boosting classifier. This allows us to reload the complete system later without rebuilding the preprocessing separately.

In [ ]:
# Create the models directory
models_dir = "/content/drive/MyDrive/CSC4110_G09/models"
os.makedirs(models_dir, exist_ok=True)

# Save the complete pipeline
model_path = os.path.join(models_dir, "G09_model.joblib")
joblib.dump(final_pipe, model_path)

print("Model saved successfully.")
print("Saved location:", model_path)

Model saved successfully.
Saved location: /content/drive/MyDrive/CSC4110_G09/models/G09_model.joblib


## 6. Prepare Default Values for the Web Application

The web application will collect seven selected employee-related inputs. However, the trained model requires all 30 features.

To address this, we calculate the median for numerical features and the most frequent value for categorical features. These values will fill the fields not entered by the user.

The defaults are saved separately in a JSON file.

In [ ]:
# Create default values for all model features
default_values = {}

# Use median values for numerical features
for col in numeric_features:
    default_values[col] = float(X[col].median())

# Use the most frequent value for categorical features
for col in categorical_features:
    default_values[col] = str(X[col].mode()[0])

# Save defaults as a JSON file
defaults_path = os.path.join(models_dir, "G09_defaults.json")

with open(defaults_path, "w") as file:
    json.dump(default_values, file, indent=4)

print("Default values saved successfully.")
print("Total default features:", len(default_values))
print("Saved location:", defaults_path)

Default values saved successfully.
Total default features: 30
Saved location: /content/drive/MyDrive/CSC4110_G09/models/G09_defaults.json


## 7. Reload and Test the Saved Model

We reload the saved machine learning pipeline using `joblib.load()`.

We then use one sample employee record to verify that the reloaded pipeline can generate an attrition prediction and probability.

This is a functional test of the saved model, not an independent evaluation of its predictive performance.

In [ ]:
# Reload the saved pipeline
loaded_model = joblib.load(model_path)

# Select one sample employee record
sample_employee = X.iloc[[0]]

# Generate prediction and probability
prediction = loaded_model.predict(sample_employee)[0]
probability = loaded_model.predict_proba(sample_employee)[0][1]

# Display results
print("Model reloaded successfully.")
print("Predicted attrition:", "Yes" if prediction == 1 else "No")
print(f"Estimated attrition probability: {probability:.2%}")

Model reloaded successfully.
Predicted attrition: No
Estimated attrition probability: 47.03%


## 8. Develop the Streamlit Web Application

We develop an interactive web application using Streamlit.

The application will:
- Display the G9 project title, group members, and dataset name.
- Collect seven selected employee-related inputs.
- Load the saved machine learning pipeline.
- Estimate attrition probability when the Predict button is clicked.
- Display a suggested supportive HR conversation.

The application is intended for educational and decision-support purposes, not for making employment decisions.

In [ ]:
# Create the app directory if it does not already exist
app_dir = "/content/drive/MyDrive/CSC4110_G09/app"
os.makedirs(app_dir, exist_ok=True)

print("App directory is ready.")
print("Location:", app_dir)

App directory is ready.
Location: /content/drive/MyDrive/CSC4110_G09/app


In [ ]:
%%writefile /content/drive/MyDrive/CSC4110_G09/app/app.py

import streamlit as st
import pandas as pd
import joblib
import json
import os

# Locate saved model and default values
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_DIR = os.path.dirname(BASE_DIR)

MODEL_PATH = os.path.join(PROJECT_DIR, "models", "G09_model.joblib")
DEFAULTS_PATH = os.path.join(PROJECT_DIR, "models", "G09_defaults.json")

# Load model and default values
model = joblib.load(MODEL_PATH)

with open(DEFAULTS_PATH, "r") as file:
    defaults = json.load(file)

# Configure application
st.set_page_config(page_title="G9 Staff Attrition Predictor")

# Project information
st.title("G9 Staff Attrition Prediction")
st.subheader("CSC 4110 - Machine Learning")

st.write("**Group:** G9")

st.write("**Group Members:**")
st.write("""
- Clare Nabwile
- Abel Morara
- Owiye Sylvia
- Jack Otuko
- Obed Kiprono
""")

st.write("**Dataset:** IBM HR Analytics Employee Attrition Dataset")

st.divider()

# Employee input section
st.header("Employee Information")

overtime = st.selectbox("OverTime", ["No", "Yes"])

job_satisfaction = st.selectbox(
    "Job Satisfaction", [1, 2, 3, 4]
)

monthly_income = st.number_input(
    "Monthly Income",
    min_value=1000,
    max_value=20000,
    value=5000
)

years_at_company = st.number_input(
    "Years at Company",
    min_value=0,
    max_value=40,
    value=5
)

distance_from_home = st.number_input(
    "Distance From Home",
    min_value=1,
    max_value=29,
    value=5
)

work_life_balance = st.selectbox(
    "Work-Life Balance", [1, 2, 3, 4]
)

business_travel = st.selectbox(
    "Business Travel",
    ["Non-Travel", "Travel_Rarely", "Travel_Frequently"]
)

# Generate prediction
if st.button("Predict Attrition Risk"):

    # Start with all 30 model features
    input_data = pd.DataFrame([defaults])

    # Replace seven default values with user inputs
    input_data["OverTime"] = overtime
    input_data["JobSatisfaction"] = job_satisfaction
    input_data["MonthlyIncome"] = monthly_income
    input_data["YearsAtCompany"] = years_at_company
    input_data["DistanceFromHome"] = distance_from_home
    input_data["WorkLifeBalance"] = work_life_balance
    input_data["BusinessTravel"] = business_travel

    # Calculate attrition probability
    probability = model.predict_proba(input_data)[0][1]

    st.subheader("Prediction Result")
    st.metric(
        "Estimated Attrition Probability",
        f"{probability:.2%}"
    )

    st.info(
        "This is a model estimate, not a certainty or a basis "
        "for employment decisions."
    )

    st.subheader("Suggested Support Conversation")

    st.write(
        "Would you like to discuss your workload, work-life balance, "
        "travel demands, or any support that could improve your "
        "experience at work?"
    )

Overwriting /content/drive/MyDrive/CSC4110_G09/app/app.py


## 9. Create Application Requirements

The Streamlit application requires specific Python libraries to run.

We create a `requirements.txt` file listing the required dependencies so that the application environment can be configured easily.

In [ ]:
%%writefile /content/drive/MyDrive/CSC4110_G09/app/requirements.txt

streamlit
scikit-learn
joblib
pandas

Overwriting /content/drive/MyDrive/CSC4110_G09/app/requirements.txt


## 10. Install and Run the Streamlit Application

In this section, we install Streamlit and launch the staff attrition prediction web application for testing.

In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 20.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 53.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


### 10.1 Launch the Streamlit Application

We launch the Streamlit application using the saved `app.py` file and expose it through Google Colab so that we can interact with and test the application.

In [ ]:
# Start the Streamlit application
!streamlit run /content/drive/MyDrive/CSC4110_G09/app/app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true &

# Wait briefly for the server to start
import time
time.sleep(5)

# Display the application in Colab
from google.colab import output
output.serve_kernel_port_as_window(8501)



2026-11-26 08:24:51.580 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.138.244.170:8501



In [ ]:
!nohup streamlit run /content/drive/MyDrive/CSC4110_G09/app/app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true > /content/streamlit.log 2>&1 &

In [ ]:
!curl -I http://127.0.0.1:8501

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8501)

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501)

In [ ]:
!ps aux | grep '[s]treamlit'

In [ ]:
!tail -40 /content/streamlit.log

In [ ]:
!curl -I --max-time 10 http://127.0.0.1:8501

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501)

In [ ]:
!tail -40 /content/streamlit.log

In [ ]:
!pkill -f "streamlit run"

In [ ]:
!nohup streamlit run /content/drive/MyDrive/CSC4110_G09/app/app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.enableWebsocketCompression false \
  > /content/streamlit.log 2>&1 &

In [ ]:
!curl -I --max-time 10 http://127.0.0.1:8501

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501)

## Task 9.1: Workflow Explanation

The project began with collecting employee information from the IBM HR Analytics Employee Attrition Dataset. The data was cleaned and prepared by removing unnecessary columns and separating the input features from the target variable. A preprocessing pipeline was developed to handle numerical and categorical features before training the Gradient Boosting classifier. The complete pipeline was saved using Joblib and reloaded to verify that it could make predictions. Finally, a Streamlit web application was developed to accept employee information and display estimated attrition risk with a suggested support conversation.

The complete pipeline was saved instead of only the model to ensure that the same preprocessing steps are applied consistently when making predictions on new data.